In [ ]:
%%capture
!pip install unsloth
# Get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

**Loading the Model**

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! it auto supports RoPE Scaling internally
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit",
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
      model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    # model_name = "mrbmaryam/LSH_ParsLite", # Mistral Fine-tuned
    # model_name = "mrbmaryam/LLAMA_LSH_ParsLite", # LLAMA Fine-tuned
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Install PyJoules for monitoring the energy consumption

In [ ]:
!pip install pyJoules

# Full Test Inference

**Prompt**

In [ ]:
# ======================================================================================
# Zero-shot and Fine-tuning Prompt
# ======================================================================================
alpaca_prompt = """You are a log parsing assistant. Your task is to identify variable elements within the logs, generalize these elements, and construct a template that represents the structure of these log messages. Please, follow the instruction and return an accurate response.


### Instruction:
Extract the template for the following log message. Replace any variable element with the placeholder '<*>'. Do not provide any explanation, only return the template.

### Input:
{}

### Response:
{}"""

# ======================================================================================
# Few-shot Prompt
# ======================================================================================
# alpaca_prompt = """You are a log parsing assistant. Your task is to identify variable elements within the logs, generalize these elements, and construct a template that represents the structure of these log messages. Please, follow the instruction and return an accurate response.

# ### Instruction:
# Extract the template for the following log message. Replace any variable element with the placeholder '<*>'. Do not provide any explanation, only return the template.

# ### Example 1:
# Log message: Error = , LOGIN chdir(/home/spelce1/UMT2K/umt2k/ckpt_umt2k_src/TEST/NEW_TEST) failed: No such file at /10.10.34.20:56374 point [/TEST/NEW_TEST], connect to proxy proxy.cse.cuhk.edu.hk:5070 to renew session (0x14f05578bd8001b)
# Extracted template: Error = , LOGIN chdir(<*>) failed: No such file at <*>:<*> point [<*>], connect to proxy <*>:<*> to renew session (<*>)

# ### Example 2:
# Log message: ciod: In packet from nodes 91.0 and node-234 (R62-M1-Nf-C:J03-U11), message code 2 is not 3 or 4294967295 (softheader=003b005b 00030000 00000001 00000000)
# Extracted template: ciod: In packet from nodes <*> and <*> (<*>:<*>), message code <*> is not <*> or <*> (softheader=<*> <*> <*> <*>)

# ### Input:
# {}

# ### Response:
# {}"""


In [ ]:
import re
FastLanguageModel.for_inference(model)

def get_template(response):

    # pattern = r"### Response:\n(.*?)</s>"
    pattern = r"### Response:\n([\s\S]*)"
    match = re.search(pattern, response)
    if match:
        extracted_text = match.group(1).strip()
        return extracted_text
    else:
        return None


def get_response(input):
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                input,
                "",
            )
        ], return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=120, use_cache=True)


    # print(f"Input: {input}" ) # for debugging

    full_response = tokenizer.batch_decode(outputs)

    print(f"Full response: {full_response}" )

    extracted_temp = get_template(full_response[0])

    # print(f"Extracted Template: {extracted_temp}" )  # for debugging

    return extracted_temp


In [ ]:
import os
import time
import pandas as pd
from pyJoules.energy_meter import measure_energy
from pyJoules.device.nvidia_device import NvidiaGPUDomain
from pyJoules.handler.csv_handler import CSVHandler

# =====================================================================
# Set up CSV logging for energy measurement
# =====================================================================
inference_energy_log = 'New_LLAMA_2S_HPC_inference_energy_log_3.csv'
csv_handler = CSVHandler(inference_energy_log)

# =====================================================================
# Define the inference function wrapped by PyJoules
# =====================================================================
@measure_energy(domains=[NvidiaGPUDomain(0)], handler=csv_handler)
def run_inference(input_file, output_file, checkpoint_file):
    """Run inference and monitor GPU energy consumption."""
    df = pd.read_csv(input_file)

    if 'Content' not in df.columns or 'EventTemplate' not in df.columns:
        raise ValueError("The input CSV must contain 'Content' and 'EventTemplate' columns.")

    if 'Dataset' not in df.columns:
        raise ValueError("The input CSV must contain a 'Dataset' column for per-dataset timing.")

    # Check for checkpoint to resume
    if os.path.exists(checkpoint_file):
        df_checkpoint = pd.read_csv(checkpoint_file)
        processed_count = len(df_checkpoint)
        print(f"Resuming from checkpoint... {processed_count} rows already processed.")
    else:
        df_checkpoint = df.copy()
        df_checkpoint["New_LLAMA_2S_HPC_3"] = ""
        df_checkpoint["Inference_time_3"] = 0.0  # <-- add inference time column
        processed_count = 0

    counter = processed_count
    dataset_times = {}  # Store inference time per dataset

    for index, row in df.iloc[processed_count:].iterrows():
        counter += 1
        log_entry = row['Content']
        dataset_name = row['Dataset']

        # Initialize time accumulator for this dataset
        if dataset_name not in dataset_times:
            dataset_times[dataset_name] = 0.0

        print(f"\nCounter: {counter}")
        print(f"Dataset: {dataset_name}")
        print(f"Log Event: {log_entry}")

        start_time = time.time()
        try:
            template = get_response(log_entry)
        except Exception as e:
            print(f"Error processing log entry: {e}")
            template = "ERROR"
        end_time = time.time()

        elapsed = end_time - start_time
        dataset_times[dataset_name] += elapsed

        print(f"Extracted Template: {template}")
        print(f"Time for this entry: {elapsed:.4f} s")

        df_checkpoint.at[index, "New_LLAMA_2S_HPC_3"] = template
        df_checkpoint.at[index, "Inference_time_3"] = elapsed  # store per-log inference time

        # Save checkpoint every 10 entries
        if counter % 10 == 0:
            df_checkpoint.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {counter}")

    # Final save after processing all logs
    df_checkpoint.to_csv(output_file, index=False)
    print("Final results saved!")

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    return len(df), dataset_times  # Return number of logs and per-dataset times


# =====================================================================
# Main execution
# =====================================================================
FastLanguageModel.for_inference(model)

input_file = "/content/New_HPC.csv"
output_file = "/content/New_LLAMA_2S_HPC_3_output.csv"
checkpoint_file = "New_LLAMA_2S_HPC_3_checkpoint.csv"

print("Starting inference with PyJoules GPU monitoring...")
start_time = time.time()

num_logs, dataset_times = run_inference(input_file, output_file, checkpoint_file)

end_time = time.time()
inference_duration = end_time - start_time
print(f"\nTotal inference finished in {inference_duration:.2f} seconds for {num_logs} log entries.")

# Save energy data
csv_handler.save_data()
print(f"Energy data saved to '{inference_energy_log}'.")

# =====================================================================
# Analyze energy data
# =====================================================================
df_energy = pd.read_csv(inference_energy_log, sep=';')
gpu_col = [c for c in df_energy.columns if 'nvidia' in c.lower() or 'gpu' in c.lower()]
gpu_col = gpu_col[0]

df_energy[gpu_col] = pd.to_numeric(df_energy[gpu_col], errors='coerce')

total_energy_millijoules = df_energy[gpu_col].sum()
total_energy_joules = total_energy_millijoules / 1000
average_power_watts = total_energy_joules / inference_duration
average_time_per_log = inference_duration / num_logs if num_logs > 0 else 0

print("\n=== Inference Energy Report ===")
print(f"GPU column used: {gpu_col}")
print(f"Total GPU Energy Consumed: {total_energy_millijoules:.4f} milli J")
print(f"Average GPU Power: {average_power_watts:.2f} W")
print(f"Total Inference Duration: {inference_duration:.2f} s")
print(f"Number of Logs: {num_logs}")
print(f"Average Inference Time per Log: {average_time_per_log:.4f} s/log")

# =====================================================================
# Log dataset summary for comparison
# =====================================================================
summary_file = "New_LLAMA_2S_HPC_3_energy_summary.csv"
input_dataset_name = os.path.basename(input_file)

# Base summary entry (total)
summary_entries = [{
    "Dataset": input_dataset_name,
    "Subset": "Total",
    "Num_Logs": num_logs,
    "Total_Inference_Time_sec": inference_duration,
    "Avg_Time_per_Log_sec": average_time_per_log,
    "Total_Energy": total_energy_joules,
    "Average_Power_W": average_power_watts
}]

# Add per-dataset runtimes
for dset, t in dataset_times.items():
    subset_df = pd.read_csv(input_file)
    subset_count = subset_df[subset_df["Dataset"] == dset].shape[0]
    summary_entries.append({
        "Dataset": input_dataset_name,
        "Subset": dset,
        "Num_Logs": subset_count,
        "Total_Inference_Time_sec": t,
        "Avg_Time_per_Log_sec": t / subset_count if subset_count > 0 else 0,
        "Total_Energy": "",
        "Average_Power_W": ""
    })

summary_df = pd.DataFrame(summary_entries)

# Append or create summary file
if os.path.exists(summary_file):
    existing = pd.read_csv(summary_file)
    summary_df = pd.concat([existing, summary_df], ignore_index=True)

summary_df.to_csv(summary_file, index=False)
print(f"\nSummary (total + per-dataset) appended to '{summary_file}'.")
